# MultiBench Part 2: AVMNIST — MFAS (Fusion Architecture Search)

In Part 1 we hand-designed the fusion. Here an algorithm **designs the fusion for us** via MFAS — Multimodal Fusion Architecture Search.

**What you'll learn**
- Why choosing *how* and *where* to fuse modalities is itself a hard problem.
- The core MFAS idea (Pérez-Rúa et al., *MFAS*, CVPR 2019): treat fusion design as a **search** over architectures instead of guessing.
- The role of the **surrogate** model — a cheap accuracy predictor that makes the search affordable.

Dataset: **AVMNIST** (image = MNIST digit, audio = spectrogram of the spoken digit); task = 10-way digit classification. The unimodal encoders are pretrained and frozen — only the fusion is searched.

Designed to run on **Google Colab**.

Requirements:
- Free space on Google Drive to clone MultiBench and save results (~2GB).
- Copy this notebook to your own Drive first (File -> "Save a copy in Drive") so your results and trained models persist.

In [2]:
# Mount Google Drive directly to /content/drive so MyDrive is at /content/drive/MyDrive/
from google.colab import drive
drive.mount("/content/drive/")

ModuleNotFoundError: No module named 'google'

## Clone or update MultiBench

In [ ]:
import os

repo_dir = "/content/drive/MyDrive/MultiBench"

if os.path.isdir(os.path.join(repo_dir, ".git")):
    print("Repo already exists, pulling latest changes...")
    !git -C "$repo_dir" fetch origin
    !git -C "$repo_dir" reset --hard origin/main
else:
    print("Cloning MultiBench...")
    !git clone https://github.com/human-ai-lab/multibench.git "$repo_dir"

print(f"Repo location: {repo_dir}")

## Setup Python path

In [ ]:
import sys

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

print(f"Using MultiBench from: {repo_dir}")

## Check and download AVMNIST data

Data is stored at `MultiBench/data/avmnist/` — inside the repo on Drive, so it persists across sessions and matches the path used by the local `.py` scripts.

In [ ]:
data_path = os.path.join(repo_dir, "data", "avmnist")

os.makedirs(os.path.join(repo_dir, "data"), exist_ok=True)

if os.path.isdir(data_path) and os.listdir(data_path):
    print(f"Data found: {data_path}")
else:
    tarball = os.path.join(repo_dir, "data", "avmnist.tar.gz")
    print("Downloading avmnist.tar.gz (one-time)...")
    !wget -q --show-progress -O "$tarball" https://filedn.eu/lDTxyzlMbdMJJq0AvECx20X/avmnist.tar.gz
    !tar -xzf "$tarball" -C "$repo_dir/data/"
    !rm "$tarball"
    print(f"Extracted to: {data_path}")

print(f"data_path = {data_path}")

## Install dependencies

In [ ]:
!pip install -q memory-profiler

## Setup device

In [ ]:
import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load AVMNIST dataset

In [ ]:
import utils.surrogate as surr
from datasets.avmnist.get_data import get_dataloader

traindata, validdata, testdata = get_dataloader(data_path, batch_size=32)

## Train MFAS

For MFAS you do not define a fusion layer or classification head manually — they are handled by the architecture search. You provide pretrained unimodal encoders and hyperparameters.

In [ ]:
from training_structures.architecture_search import train

image_enc = os.path.join(repo_dir, "pretrained", "avmnist", "image_encoder.pt")
audio_enc = os.path.join(repo_dir, "pretrained", "avmnist", "audio_encoder.pt")

s_data = train(
    [image_enc, audio_enc],
    16,                          # rep_size: width of each encoder's output
    10,                          # number of classes (digits 0-9)
    [(6, 12, 24), (6, 12, 24, 48, 96)],  # per-layer outputs of each encoder
                                 # (image, then audio) -> defines fusable features
    traindata,
    validdata,
    surr.SimpleRecurrentSurrogate().to(device),  # surrogate accuracy predictor
    (3, 5, 2),                   # search space of the fusion layer
    epochs=6                     # epochs to partially train each candidate
)

That concludes Part 2! See Part 3 for an example using MultiBench's MCTN paradigm.